# Runnables

## Goal 

understandingL:

- What a Runnables actually is in LangChain
- Why LangChain introduced the Runnables abstraction.
- What problem Runnables solve.
- Why different LangChain components can be composed together.
- The common interface shared by Runnables.
- How `invoke()`, `batch()`, `stream()` and `ainvoke()` relate to the Runnable interface.
- The different between a Runnable and a RunnableSequence.
- How different Runnable types can be combined to build production pipelines.
- How Runnables will become the foundation for more advanced concepts such as tools, agents and LangGraph.

In [1]:
# Load the environment
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Read the model name
import os
MODEL_NAME = os.environ["GEMINI_MODEL"]
API_KEY = os.environ["GOOGLE_GENERATIVE_AI_API_KEY"]

In [3]:
# Create the LangChain model
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI (model = MODEL_NAME, api_key =API_KEY, temperature = 0)

In [6]:
# Create a Runnable from a normal function
from langchain_core.runnables import RunnableLambda

def double(x):
    return x * 2

double_runnable = RunnableLambda(double)

In [8]:
# Invoke the new runnable function
double_result = double_runnable.invoke(5)
print(double_result)
type(double_runnable)

10


langchain_core.runnables.base.RunnableLambda

In [13]:
# Define the parser
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [14]:
# Create the function then create the RUnnable
def add_prefix(text):
    return "AI: " + text

prefix_runnable = RunnableLambda(add_prefix)


In [ ]:
# Create and invoke the chain
chain = parser | prefix_runnable

print(chain.invoke("LangChain is useful."))

AI: LangChain is useful.


In [ ]:
# batch with RunnableLambda

def double (x):
    return x * 2

double_runnable = RunnableLambda(double)

results = double_runnable.batch([1,2,3,4,5,6])

print(type(results))
print(results)

print(type(double_runnable))

<class 'list'>
[2, 4, 6, 8, 10, 12]
<class 'langchain_core.runnables.base.RunnableLambda'>


In [18]:
# Chunk with RunnableLambda
for chunk in double_runnable.stream(5):
    print(type(chunk))
    print(repr(chunk))

<class 'int'>
10


In [ ]:
def introduce (data):
    return f"{data['name']} is {data['age']} years old."


In [21]:
introduce_runnable = RunnableLambda(introduce)

result = introduce_runnable.invoke(
    {
        "name": "Mohamed",
        "age": 30
    }
)

print(result)
print(type(result))

Mohamed is 30 years old.
<class 'str'>


In [24]:
# async RunnableLambda

async def double_async(x):
    return x * 2

double_runnable_async = RunnableLambda(double_async)

result = await double_runnable_async.ainvoke(5)

print(result)
print(type(double_runnable_async))

10
<class 'langchain_core.runnables.base.RunnableLambda'>


In [ ]:
# RunnablePassthrough

from langchain_core.runnables import RunnablePassthrough

passthrough = RunnablePassthrough() # Receive an input and pass that same input through unchanged.

result = passthrough.invoke("Hello")

print (result)

Hello


In [ ]:
result = passthrough.invoke(
    {
        "question": "What is LangChain?",
        "topic":"AI"
    }
)

print(type(result))
print(result)

<class 'dict'>
{'question': 'What is LangChain?', 'topic': 'AI'}


In [ ]:
# RunnableParallel

from langchain_core.runnables import RunnableParallel
def fake_retrieve(question):
    return["Document about "+ question]

retriever = RunnableLambda(fake_retrieve)

chain = RunnableParallel( # RunnableParallel combines branch outputs into one dictionary
    {
        "question": RunnablePassthrough(), # When invoke it will return the same input inside the invoke
        "context": retriever
    }
)

In [21]:
result = chain.invoke("What is LangChain?")
print(result)

{'question': 'What is LangChain?', 'context': ['Document about What is LangChain?']}


```text
                  "What is LangChain?"
                         │
              ┌──────────┴──────────┐
              ↓                     ↓
 RunnablePassthrough          retriever
              ↓                     ↓
          question              context
              │                     │
              └──────────┬──────────┘
                         ↓
                    final dict
```

In [24]:
def add_ten(x):
    return x +10

add_ten_runnable = RunnableLambda(add_ten)

parallel = RunnableParallel(
    {
        "original":RunnablePassthrough(),
        "doubled":RunnableLambda(double),
        "plus_ten": add_ten_runnable
    }
)

In [25]:
parallel.invoke(5)

{'original': 5, 'doubled': 10, 'plus_ten': 15}